# RZLT Training Model
Binary classifier to predict whether a parcel is a **live RZLT site** (`Y=1`) or **non-taxed zoned land** (`N=0`).

**Pipeline:**
1. Feature Extraction (via `Feature_Extractor.py`)
2. Train/Test Split (Stratified)
3. Train: Random Forest + XGBoost
4. Evaluate: Classification Report, ROC-AUC, Confusion Matrix
5. Explainability: SHAP Feature Importance

## Cell 1 — Imports & Feature Extraction

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy.spatial import cKDTree
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

# ── CONFIG ─────────────────────────────────────────────────────────────────────
INPUT_FILE  = "Datasets/RZLT_GZT_Merged/CWLA_Merged.geojson"   # ← change LA here
GTFS_STOPS  = "Datasets/GTFS_All/stops.txt"
RANDOM_SEED = 42

print(f"Input file : {INPUT_FILE}")
print(f"Random seed: {RANDOM_SEED}")

## Cell 2 — Load & Validate Dataset

In [ ]:
# Load merged GeoJSON and reproject to ITM
gdf = gpd.read_file(INPUT_FILE).to_crs(epsg=2157)

# Detect file type
is_merged = gdf['LIVE_SITE'].nunique() > 1
print(f"File type   : {'GZT-Merged (Y+N)' if is_merged else 'RZLT-only (Y only) ⚠️ Cannot train binary classifier'}")
print(f"Total rows  : {len(gdf)}")
print(f"\nClass distribution:")
print(gdf['LIVE_SITE'].value_counts())
print(f"\nClass ratio (Y:N) = {gdf['LIVE_SITE'].value_counts()['Y']} : {gdf['LIVE_SITE'].value_counts()['N']}")

if not is_merged:
    raise ValueError("Dataset must contain both Y and N classes. Re-run GZT&RZLT Merge.ipynb first.")

print(f"\nNull counts in key columns:")
print(gdf[['SITE_AREA', 'ZONE_GZT', 'LOCAL_AUTHORITY', 'LIVE_SITE']].isnull().sum())

## Cell 3 — Feature Engineering

In [ ]:
# ── SITE AREA ──────────────────────────────────────────────────────────────────
if 'SITE_AREA' not in gdf.columns or gdf['SITE_AREA'].isnull().any():
    gdf['SITE_AREA'] = gdf.geometry.area / 10000.0

# ── TRANSIT PROXIMITY (cKDTree nearest-neighbour) ──────────────────────────────
print("Computing transit distances...")
stops_df = pd.read_csv(GTFS_STOPS)
stops_gdf = gpd.GeoDataFrame(
    stops_df,
    geometry=gpd.points_from_xy(stops_df.stop_lon, stops_df.stop_lat),
    crs="EPSG:4326"
).to_crs(epsg=2157)

stop_coords   = np.column_stack([stops_gdf.geometry.x, stops_gdf.geometry.y])
parcel_coords = np.column_stack([gdf.geometry.centroid.x, gdf.geometry.centroid.y])
distances, _  = cKDTree(stop_coords).query(parcel_coords)
gdf['DISTANCE_TO_TRANSIT'] = distances

print(f"  Avg distance to transit: {distances.mean():.0f} m")
print(f"  Min: {distances.min():.0f} m  |  Max: {distances.max():.0f} m")

# ── ENCODE CATEGORICAL FEATURES ───────────────────────────────────────────────
num_cols = ['SITE_AREA', 'DISTANCE_TO_TRANSIT']
cat_cols = ['ZONE_GZT', 'LOCAL_AUTHORITY']

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded       = ohe.fit_transform(gdf[cat_cols])
cat_feature_names = list(ohe.get_feature_names_out(cat_cols))

X = np.hstack([gdf[num_cols].values, cat_encoded])
y = (gdf['LIVE_SITE'] == 'Y').astype(int).values
feature_names = num_cols + cat_feature_names

print(f"\nFeature matrix : {X.shape}  ({len(feature_names)} features)")
print(f"Target         : Y={y.sum()}  N={(y==0).sum()}")
print(f"\nFeatures: {feature_names}")

## Cell 4 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y          # preserves Y/N ratio in both splits
)

print(f"Training set : {X_train.shape[0]} rows  → Y={y_train.sum()}  N={(y_train==0).sum()}")
print(f"Test set     : {X_test.shape[0]}  rows  → Y={y_test.sum()}   N={(y_test==0).sum()}")
print(f"\ny_train classes: {np.unique(y_train)}")
print(f"y_test  classes: {np.unique(y_test)}")

## Cell 5 — Train Classifiers

In [ ]:
# ── CLASS WEIGHT for imbalance ─────────────────────────────────────────────────
neg, pos = (y_train == 0).sum(), y_train.sum()
scale_pos_weight = neg / pos   # used by XGBoost
print(f"scale_pos_weight (neg/pos): {scale_pos_weight:.3f}")

# ── RANDOM FOREST ──────────────────────────────────────────────────────────────
print("\nTraining Random Forest...")
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight='balanced',    # handles Y/N imbalance
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rf_clf.fit(X_train, y_train)
print("  Random Forest ✓")

# ── XGBOOST ───────────────────────────────────────────────────────────────────
print("Training XGBoost...")
xgb_clf = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
xgb_clf.fit(X_train, y_train)
print("  XGBoost ✓")

## Cell 6 — Evaluation

In [ ]:
rf_preds  = rf_clf.predict(X_test)
xgb_preds = xgb_clf.predict(X_test)

rf_probs  = rf_clf.predict_proba(X_test)[:, 1]
xgb_probs = xgb_clf.predict_proba(X_test)[:, 1]

print("=" * 55)
print("RANDOM FOREST — Classification Report")
print("=" * 55)
print(classification_report(y_test, rf_preds, target_names=['N (non-taxed)', 'Y (live site)']))
print(f"ROC-AUC: {roc_auc_score(y_test, rf_probs):.4f}")

print()
print("=" * 55)
print("XGBOOST — Classification Report")
print("=" * 55)
print(classification_report(y_test, xgb_preds, target_names=['N (non-taxed)', 'Y (live site)']))
print(f"ROC-AUC: {roc_auc_score(y_test, xgb_probs):.4f}")

## Cell 7 — Confusion Matrices & ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Confusion Matrices ─────────────────────────────────────────────────────────
for ax, preds, title in zip(
    axes,
    [rf_preds, xgb_preds],
    ['Random Forest', 'XGBoost']
):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, preds),
        display_labels=['N (non-taxed)', 'Y (live site)']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── ROC Curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for probs, label, color in [
    (rf_probs,  f'Random Forest (AUC={roc_auc_score(y_test, rf_probs):.3f})',  'steelblue'),
    (xgb_probs, f'XGBoost       (AUC={roc_auc_score(y_test, xgb_probs):.3f})', 'darkorange')
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax.plot(fpr, tpr, label=label, linewidth=2, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random baseline')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — RZLT Classifier', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 8 — Cross-Validation (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print("Running 5-fold cross-validation...")

rf_cv  = cross_val_score(rf_clf,  X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
xgb_cv = cross_val_score(xgb_clf, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"\nRandom Forest CV ROC-AUC : {rf_cv.mean():.4f}  ± {rf_cv.std():.4f}")
print(f"XGBoost       CV ROC-AUC : {xgb_cv.mean():.4f}  ± {xgb_cv.std():.4f}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 4))
models  = ['Random Forest', 'XGBoost']
means   = [rf_cv.mean(), xgb_cv.mean()]
stds    = [rf_cv.std(),  xgb_cv.std()]
colors  = ['steelblue', 'darkorange']

bars = ax.bar(models, means, yerr=stds, capsize=6, color=colors, alpha=0.85, width=0.4)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('ROC-AUC', fontsize=12)
ax.set_title('5-Fold Cross-Validation ROC-AUC', fontsize=13, fontweight='bold')
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, mean + 0.01, f'{mean:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 9 — SHAP Feature Importance (XGBoost)

In [ ]:
import shap

print("Computing SHAP values for XGBoost...")
explainer   = shap.TreeExplainer(xgb_clf)
shap_values = explainer(X_test)
shap_values.feature_names = feature_names

# ── 1. Summary (Beeswarm) ─────────────────────────────────────────────────────
print("\n[SHAP] Summary Plot — Global Feature Impact")
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=True)

# ── 2. Bar Chart of Mean |SHAP| ───────────────────────────────────────────────
print("\n[SHAP] Feature Importance Bar Chart")
shap.plots.bar(shap_values, max_display=15, show=True)

# ── 3. Dependence plots for top spatial features ──────────────────────────────
print("\n[SHAP] Dependence Plot — DISTANCE_TO_TRANSIT")
shap.dependence_plot('DISTANCE_TO_TRANSIT', shap_values.values, X_test,
                     feature_names=feature_names)

print("\n[SHAP] Dependence Plot — SITE_AREA")
shap.dependence_plot('SITE_AREA', shap_values.values, X_test,
                     feature_names=feature_names)

## Cell 10 — Save Models

In [ ]:
import joblib
import os

os.makedirs('Models', exist_ok=True)

# Save models and encoder
joblib.dump(rf_clf,       'Models/rf_classifier.pkl')
joblib.dump(xgb_clf,      'Models/xgb_classifier.pkl')
joblib.dump(ohe,          'Models/ohe_encoder.pkl')
joblib.dump(feature_names,'Models/feature_names.pkl')

print("Models saved to /Models/:")
print("  rf_classifier.pkl")
print("  xgb_classifier.pkl")
print("  ohe_encoder.pkl")
print("  feature_names.pkl")